In [ ]:
# -*- coding: utf-8 -*-
"""
Scenario-at-a-Time IIR Pipeline (variant)
-----------------------------------------
This is a structural variant of master_iir_workflow.py. It generates
simulated student responses ONE SCENARIO AT A TIME instead of all three
in a single call. Each student therefore costs 3 API calls instead of 1.

Purpose: produce data comparable to the current all-at-once pipeline so we
can empirically test whether the structural difference matters.

KEY DIFFERENCES vs. the original pipeline:
  - Three API calls per student, one per scenario.
  - Each call only sees ONE story and its 4 questions.
  - The model has no knowledge of the other scenarios when answering.
  - Within-scenario, the (a)/(b) question pair shuffling is preserved.
  - Across-scenario shuffling still happens (random order of Cat/Lying/Stealing).

CACHING NOTE:
  Vertex/Gemini implicit caching has minimum token thresholds (~4096 tokens
  for Gemini, last I checked — verify against current docs). Your system
  prompt + instructions are likely under that threshold, in which case
  caching produces zero savings. We pass `cached_content` only if it would
  plausibly help; otherwise we send normally.

  This script does NOT use explicit context caching (CachedContent objects).
  Implicit caching is the safer default — it's automatic, no setup, no risk
  of paying for a cache that never gets hit.

WHAT IS UNCHANGED FROM THE ORIGINAL:
  - Waypoint definitions, system prompt, output schema.
  - Format pool balancing (~half comic, half text per waypoint).
  - Resume logic via progress.json.
  - Per-student seed determinism (so the same student in both pipelines
    gets the same waypoint, same format, same scenario order).
"""

import os
import json
import time
import random
import pandas as pd
import vertexai
from vertexai.generative_models import GenerativeModel, Part, GenerationConfig

# -----------------------------------------------------------------------------
# Setup
# -----------------------------------------------------------------------------

PROJECT_CODE = os.environ.get("PROJECT_CODE")

vertexai.init(project=PROJECT_CODE, location="global")
OUTPUT_DIR = "IIR_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME = "gemini-3.1-pro-preview"

# Progress file for resume logic — mirrors master_iir_workflow.py's pattern.
# A separate filename so the two pipelines don't stomp on each other's progress.
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "scenario_at_a_time_progress.json")

# -----------------------------------------------------------------------------
# Profiles, system prompt, story texts, comic paths
#
# These should mirror your master_iir_workflow.py EXACTLY. To keep this script
# self-contained I'm declaring them here, but in practice you should consider
# importing them from a shared module so the two pipelines can't drift apart.
# -----------------------------------------------------------------------------
profiles = {
    "WAYPOINT0": {
        "label": "0 — No inference",
        "definition": (
            "Students do not generate an inference nor try to answer the question.  Instead, they say 'I don't know' or make a nonsensical statement (e.g., stating irrelevant world knowledge, unrelated to the story). In other words, they fail to make a coherent link  between the question, the text, or their world knowledge."
        )
    },
    "WAYPOINT1": {
        "label": "1 —Text-Implicit",
        "definition": (
            "Students generate an inference connecting the question only with the information explicitly stated in the story. Use of world-knowledge is absent. This signals student inference is limited to text-base representation; without the schema/background knowledge activation, their inference may be thin, lacking elaboration of the “why” and the “how” behind the narrative."

        )
    },
    "WAYPOINT2": {
        "label": "2 — Script-Implicit",
        "definition": (
            "Students generate an inference connecting the question primarily with their background knowledge. Use of text-base information is implicit. This signals their 'script-based' reasoning—they answer based on how the world usually operates without explicitly integrating how the story specifically describes it."
        )
    },
    "WAYPOINT3": {
        "label": "3 — Combination",
        "definition": (
             "Students generate an inference connecting the question with both world/background knowledge AND information explicitly mentioned in the text (text-based information). In other words, students go beyond merely identifying text-base information,  anchoring it within a broader real-world schema (i.e., background knowledge) to explain the why or how behind the narrative. This achieves the highest level of coherence (“the situation model”, Kintsch, 1988, 1998) because the student integrates the text-base information and their prior knowledge to generate a novel, unified interpretation."

        )
    },
}

# -----------------------------------------------------------------------------
# Experience levels — 4-level scheme matching the original System_Prompt
#
# These are the same 4 levels referenced in the System_Prompt
# ("No experience" through "A lot"). Now they're actually injected into the
# user prompt rather than just mentioned in the system prompt.
#
# INDEPENDENT of the waypoint. Waypoint dictates depth and kind of inference;
# experience affects style, vocabulary, fluency, engagement — NOT reasoning depth.
#
# Format-conditional phrasing: for comic students this is "comic experience";
# for text students this is "reading comprehension experience". Same level
# structure, different framing — applied via FORMAT_PHRASING below.
# -----------------------------------------------------------------------------
experience_levels = {
    "none": {
        "label": "No experience",
        "definition": (
            "The student has essentially no prior experience with this kind "
            "of material. Responses are short and unpolished. Vocabulary is "
            "very simple. Sentences are brief and may be incomplete. The "
            "student may seem unsure how to engage with the task itself."
            "NOTE: this does NOT change reasoning depth — that is "
            "determined by the waypoint."
        )
    },
    "little": {
        "label": "A little experience",
        "definition": (
            "The student has a small amount of prior experience. Responses "
            "are still relatively short and informal. Some hesitation in "
            "expression. Vocabulary is simple but functional."
             "NOTE: this does NOT change reasoning depth — that is "
            "determined by the waypoint."
        )
    },
    "moderate": {
        "label": "More than a little, less than a lot",
        "definition": (
            "The student has moderate experience. Responses are reasonably "
            "fluent and developed but not highly polished. Comfortable "
            "engaging with the material. May not always elaborate fully."
            " NOTE: this does NOT change reasoning depth — that is "
            "determined by the waypoint."
        )
    },
    "lot": {
        "label": "A lot",
        "definition": (
            "The student has substantial experience. Responses are fluent "
            "and articulate. Vocabulary is varied. Ideas are expressed with "
            "clarity. NOTE: this does NOT change reasoning depth — that is "
            "determined by the waypoint."
        )
    },
}

# How to phrase "experience" depending on which format the student got.
FORMAT_PHRASING = {
    "comic": "COMIC EXPERIENCE",
    "text":  "READING COMPREHENSION EXPERIENCE",
}

System_Prompt = ("You are an experienced high school / middle school instructor with deep expertise in how students develop Inferential Integration in Reading (IIR) — "
"the ability to understand information from a story with world knowledge to construct meaning. "
"Based on your teaching experience, predict the answer that a specific student would give, based on their respective performance waypoint. "
"Do not explain your reasoning. Only output the student's response."
"\n Each student response must follow one of the 4 descriptive waypoints regarding student performance:"
"\n WAYPOINT0: No meaningful response provided."
    "\n Students do not generate an inference nor try to answer the question.  Instead, they say that they don't know or make a nonsensical statement (e.g., stating irrelevant world knowledge, unrelated to the story). In other words, they fail to make a coherent link  between the question, the text, or their world knowledge."
"\n WAYPOINT1: Text-Base Only (Local)"
    "\n Students generate an inference connecting the question only with the information explicitly stated in the story. Use of world-knowledge is absent. This signals student inference is limited to text-base representation; without the schema/background knowledge activation, their inference may be thin, lacking elaboration of the “why” and the “how” behind the narrative"
"\n WAYPOINT2: World Knowledge (Global) Only"
    "\n Students generate an inference connecting the question primarily with their background knowledge. Use of text-base information is implicit. This signals their 'script-based' reasoning—they answer based on how the world usually operates without explicitly integrating how the story specifically describes it."
"\n WAYPOINT3: Combination (Global and Local)"
    "\n Students generate an inference connecting the question with both world/background knowledge AND information explicitly mentioned in the text (text-based information). In other words, students go beyond merely identifying text-base information,  anchoring it within a broader real-world schema (i.e., background knowledge) to explain the why or how behind the narrative. This achieves the highest level of coherence (“the situation model”, Kintsch, 1988, 1998) because the student integrates the text-base information and their prior knowledge to generate a novel, unified interpretation."
"\n\n Note: Each simulated student will either evaluate a comic OR text-only scenario that will be given by the user prompt."

"\n Each simulated student also will specify one of the 4 'comic experience levels' that that they have, given by the user prompt."
"\n The comic experience levels are:"
"\n 'No experience', 'A little experience', 'More than a little / less than a lot', 'A lot' "
"\n The students will also give brief responses."
"\n The students will answer two pairs of questions: (1a)/(1b) and (2a)/(2b). Be sure that each pair maintains contintuity or relevance between the (a) and (b) parts, while also allowing some flexibility for the student to potentially give creative answers as well. Follow the waypoints"
)

# Story texts — paste from your master_iir_workflow.py
cat_text     = """
There was a cat that lived on the streets that really wanted to find a home.
But nobody wanted him. “Stinky cat”, said one boy to the cat.  “Yucky cat”, said a girl to the cat.

Then on one cloudy day, the cat walked down a different street.
He saw a boy with red hair sitting by himself, crying.  The cat wanted to make him feel better.

So the cat jumped up on the bench, and walked onto his lap.
The cat looked straight up at the red headed boy and said “Meow”.

The boy looked down at the dirty cat. The boy pet the cat.
The cat said “purrrrr”.  Then some boys (KIDS)  started laughing at the red headed boy.
“Hahaha.  Now you’re just as yucky as that dirty cat” shouted one of the boys.

The boy hated being laughed at, so he quickly put the cat down and stopped petting the cat.
He shouted at the cat, “go away cat!  You’re making them laugh at me!”
“Hey, you can come back and play basketball with us again, but after you wash your hands” shouted a boy.

(THEN) The red headed boy, and his friends, walked back home together and came across the cat again.
As the cat was sitting in the rain, wet, cold, and shaking, the boys were being mean to it and laughed at the cat.
“Haha it’s about time you get a shower you stinky cat!” said one of the boys.
The red headed boy saw how sad the cat was.  He walked over to it.
“Don’t think I’ll let you play basketball with us again if you touch that dirty cat”, said one of the boys.
The red headed boy bent down, and pet the cat anyways.
He said to the cat, “hey there real friend.  My name is Bruno, and I’m going to take you to your new home, my home”.


"""
lying_text   = """
Two girls, Sarah and Rachel, have been best friends since they were toddlers.
They are now in middle school, and are taking the same math class together.

•“Hey Sarah, have you studied for that math test yet?” asked Rachel.
“Yeah.  I’ve studied a lot for it.  What about you?” asked Sarah.

“I haven’t studied at all.  But I feel confident that I will do well anyways”, said Rachel.

It is the day of the test now.  They are sitting at their desks.
“Alright everyone, I am about to pass your tests out.  Remember, no cheating.
You have 40 minutes to finish”, says the teacher.

Sarah says to herself,  “Hmm I better make sure I read each question carefully, so that I don’t miss anything, and answer any questions wrong”.

Sarah looks over at her best friend Rachel.  She sees Rachel bubbling in every answer quickly on her answer sheet. Sarah says to herself, “But how could she know the answer to each question so quickly?  She didn’t even study.  She’s just probably guessing then.”

Then Sarah sees Rachel looking over the shoulder of the kid who is sitting right in front of her. It looks like she is copying his answer sheet.

Rachel notices that Sarah is looking at her.  Rachel whispers to Sarah, “Sarah shhhhh.  Don’t tell on me.  I’m your best friend”.

“Is something going on Sarah?” asks the teacher.  “Nothing is going on Ms. Craig,” answers Sarah.  Sarah continues taking the test without looking up again.


"""
stealing_text = """
Billy loved bikes.  He was riding his bike to the beach one day, when he came across the most beautiful bike he had ever seen.  He said to the boy that was riding the beautiful bike, “wow, that’s such a beautiful bike!”  “Thanks!” said the boy.

“Hey, how much did you pay for that bike?” asked Billy.  “It was $300”, answered the boy.

Billy rode his bike home.  He thought to himself, “hmmm, maybe my parents will buy me that bike for my birthday”.

Billy got home. He says to his parents,“hey mom and dad.  For my birthday I would like a new bike pretty please!”.  “No Billy.  You already have a nice bike and you don’t need a new expensive one”, said his father.

Billy felt sad, and mad.  So he took his bike, and rode it to an ice cream shop.

Billy walked out of the ice cream shop with ice cream, and found a table to sit outside. “Ice cream, you always make me feel better” Billy said to himself.  Next to him was another table, where a lady was sitting.

As Billy was eating his ice cream, the lady left and forgot her wallet.  Billy went and grabbed her wallet.  He opened her wallet.  “Wow, is this really $300? With that money, I could buy that red bike!”  Billy said.

The lady that was sitting at the table next to him came back, looking for her wallet.  She said, “hello there, have you seen my wallet?  I can’t find it anywhere.”  “No, I don’t know where it is”, said Billy.


"""

# Instructions block — copied verbatim from master_iir_workflow.py.
# Item 7 (continuity between (a) and (b)) still applies in the per-scenario
# context because (a)/(b) pairs are answered together in each call.
# Items 5 and 8 also apply unchanged.
instructions = """
==================== INSTRUCTIONS ====================
1. You MUST answer every single item. Do not leave any key empty.
2. Write exactly as a student at the specified IIR level would write.
3. Keep responses short and natural — like a real student, not an essay and only a couple of sentences.
4. Do not use perfect grammar or overly sophisticated vocabulary.
5. The meta-reasoning answers (2a, 2b) should reference BOTH the story
   AND personal experience depending on the IIR level.
6. Do not mention the IIR level anywhere.
7. Maintain continuity between part (a) and (b) for each pair of questions, that is appropriate for each waypoint level.
8. Have the student generated responses be as thorough as what is necessary to follow the waypoint guidelines.
9. Output valid JSON only — no extra text, no markdown, no code fences.
"""


def build_profile_prompt(level, experience_level=None, format_type=None):
    """Profile block — copied verbatim from master_iir_workflow.py, extended
    with an optional experience-level block.

    The experience block uses format-conditional phrasing:
      - For comic students: "COMIC EXPERIENCE"
      - For text students:  "READING COMPREHENSION EXPERIENCE"

    The waypoint takes precedence — experience only affects style/fluency.
    """
    profile = profiles[level]
    block = (
        f"==================== TARGET STUDENT PROFILE ====================\n"
        f"IIR Developmental Level: {profile['label']}\n"
        f"CONSTRUCT MAP DEFINITION:\n"
        f"{profile['definition']}\n"
    )

    if experience_level is not None:
        if format_type not in FORMAT_PHRASING:
            raise ValueError(
                f"format_type must be one of {list(FORMAT_PHRASING.keys())} "
                f"when experience_level is provided. Got: {format_type!r}"
            )
        exp = experience_levels[experience_level]
        header = FORMAT_PHRASING[format_type]
        block += (
            f"\n"
            f"{header}: {exp['label']}\n"
            f"{exp['definition']}\n"
            f"\n"
            f"IMPORTANT: The experience level above ONLY affects the student's "
            f"writing style, vocabulary, and fluency. It does NOT change the "
            f"reasoning depth or inference type — that is fully determined by "
            f"the IIR Developmental Level above. A 'No experience' WAYPOINT3 student "
            f"still integrates text and world knowledge; they just express it "
            f"in simpler language. An 'A lot' WAYPOINT1 student still uses only "
            f"text-base information; they just express it more fluently.\n"
        )

    return block


# scenarios_data — uses the SAME structure as master_iir_workflow.py:
# pairs are dicts with 'questions' (pre-formatted string) and 'keys' (list).
scenarios_data = {
    "Cat": {
        "text": cat_text,
        "comic": "Cat.pdf",
        "pairs": [
            {
                "questions": "(1a).Why did the red headed boy keep the cat?\n(1b).What made you think of that answer?",
                "keys": ["cat_1", "cat_2"]
            },
            {
                "questions": "(2a).What lesson could someone learn from this story?\n(2b).What made you think of that answer?",
                "keys": ["cat_3", "cat_4"]
            }
        ]
    },
    "Lying": {
        "text": lying_text,
        "comic": "Lying.pdf",
        "pairs": [
            {
                "questions": "(1a).Why doesn’t Sarah tell the teacher that Rachel is looking at another student’s answers?\n(1b).What made you think of that answer?",
                "keys": ["lying_1", "lying_2"]
            },
            {
                "questions": "(2a).What lesson could someone learn from this story?\n(2b).What made you think of that answer?",
                "keys": ["lying_3", "lying_4"]
            }
        ]
    },
    "Stealing": {
        "text": stealing_text,
        "comic": "Stealing.pdf",
        "pairs": [
            {
                "questions": "(1a).Why did Billy keep the lady’s wallet?\n(1b).What made you think of that answer?",
                "keys": ["stealing_1", "stealing_2"]
            },
            {
                "questions": "(2a).What lesson could someone learn from this story?\n(2b).What made you think of that answer?",
                "keys": ["stealing_3", "stealing_4"]
            }
        ]
    }
}


# -----------------------------------------------------------------------------
# Per-scenario prompt builder
#
# This mirrors master_iir_workflow.py's build_prompt() as faithfully as possible,
# with the only structural change being: ONE scenario per call instead of three.
# Header style, instruction block, profile block, and JSON schema formatting
# all match the original verbatim. The orienting sentence is adapted for the
# single-scenario context.
# -----------------------------------------------------------------------------
def build_single_scenario_prompt(level: str, scenario_name: str, format_type: str,
                                 pair_seed: int, experience_level: str = None):
    """Build the user prompt for one scenario. Returns (System_Prompt, prompt_user, keys)
    matching the original build_prompt() return shape so call_api can be used
    interchangeably.

    The pair_seed determines the (a)/(b) pair order shuffle within this
    scenario, so it's reproducible.

    experience_level: optional 'low' | 'medium' | 'high'. If provided, an
    experience block is added to the profile prompt with explicit instructions
    that experience only affects style, not reasoning depth.
    """
    s_data = scenarios_data[scenario_name]

    # Shuffle pairs deterministically using a per-call seed
    pairs_rng = random.Random(pair_seed)
    pairs = list(s_data["pairs"])
    pairs_rng.shuffle(pairs)

    # Build questions block and key list — mirrors original line 352-354
    shuffled_questions = f"{pairs[0]['questions']}\n{pairs[1]['questions']}"
    json_keys_ordered = []
    json_keys_ordered.extend(pairs[0]['keys'])
    json_keys_ordered.extend(pairs[1]['keys'])

    # Build per-scenario block — mirrors original line 356-364
    scenarios_prompt_text = f"==================== SCENARIO: {scenario_name.upper()} ====================\n"

    document_parts = []
    if format_type == "text":
        scenarios_prompt_text += f"STORY:\n{s_data['text']}\n\n"
    elif format_type == "comic":
        with open(s_data['comic'], "rb") as f:
            document_parts.append(Part.from_data(data=f.read(), mime_type="application/pdf"))

    scenarios_prompt_text += f"QUESTIONS:\n{shuffled_questions}\n\n"

    # Dynamic JSON schema — mirrors original line 366-377
    json_format_lines = [f'    "{key}": "..."' for key in json_keys_ordered]
    json_format_str = "{\n" + ",\n".join(json_format_lines) + "\n}"

    dynamic_response_format = f"""
==================== RESPONSE FORMAT ====================
Return a single valid JSON object with exactly these {len(json_keys_ordered)} keys in the exact order shown below.
Each value should be a short open-ended text response as the student would write it.
Do not include any explanation or text outside the JSON.

{json_format_str}
"""

    # Orienting sentence adapted for single-scenario context.
    if format_type == "text":
        orienting = "Below is one scenario. Please read the story and answer the questions."
        prompt_text = (
            build_profile_prompt(level, experience_level, format_type) + "\n\n" +
            orienting + "\n\n" +
            scenarios_prompt_text +
            dynamic_response_format + "\n\n" +
            instructions
        )
        prompt_user = prompt_text

    elif format_type == "comic":
        orienting = (
            f"Below is one comic story attached as a document ({scenario_name}). "
            "Please read the comic and answer the corresponding questions."
        )
        prompt_text = (
            build_profile_prompt(level, experience_level, format_type) + "\n\n" +
            orienting + "\n\n" +
            scenarios_prompt_text +
            dynamic_response_format + "\n\n" +
            instructions
        )
        prompt_user = document_parts + [prompt_text]

    return System_Prompt, prompt_user, json_keys_ordered


# -----------------------------------------------------------------------------
# API call with retry
# -----------------------------------------------------------------------------
def call_api(user_payload, max_retries=3):
    """Single API call with exponential backoff retry.

    NOTE on caching: We rely on Vertex's IMPLICIT caching, which automatically
    detects identical prompt prefixes across calls. The system_instruction is
    identical across every call in this pipeline, so it's an ideal candidate
    for implicit caching. We don't pay anything to enable it — Vertex either
    caches the prefix automatically or doesn't, depending on its threshold.
    """
    model = GenerativeModel(
        model_name=MODEL_NAME,
        system_instruction=System_Prompt,
    )

    last_err = None
    for attempt in range(max_retries):
        try:
            resp = model.generate_content(
                user_payload,
                generation_config=GenerationConfig(
                    temperature=1.0,
                    response_mime_type="application/json",
                ),
            )
            return resp.text
        except Exception as e:
            last_err = e
            wait = 2 ** attempt
            print(f"  API error (attempt {attempt+1}): {e}. Waiting {wait}s...")
            time.sleep(wait)
    raise last_err


def parse_response(raw_text, expected_keys):
    """Parse JSON; missing keys -> None."""
    try:
        data = json.loads(raw_text)
    except (json.JSONDecodeError, TypeError):
        return {k: None for k in expected_keys}
    return {k: data.get(k, None) for k in expected_keys}


# -----------------------------------------------------------------------------
# Per-student generation: 3 calls, one per scenario
#
# This function supports partial completion: if `existing_state` already
# contains responses for some scenarios, those are skipped and only the
# remaining scenarios are generated. This is what allows resume logic to
# work at the per-scenario level (not just per-student).
# -----------------------------------------------------------------------------
ALL_KEYS = [f"{s.lower()}_{i}" for s in ["cat", "lying", "stealing"] for i in range(1, 5)]


def generate_one_student(student_idx: int, level: str, format_type: str,
                         master_seed: int, experience_level: str = None,
                         existing_state=None,
                         on_scenario_done=None):
    """Generate all 12 answers for one student via up to 3 separate API calls.

    Args:
        student_idx: 0-based index of this student.
        level: IIR waypoint string (e.g., "WAYPOINT1").
        format_type: "comic" or "text".
        master_seed: Used to derive a deterministic per-student RNG.
        experience_level: 'low' | 'medium' | 'high' | None. Independent of
            waypoint; affects style only, not reasoning depth.
        existing_state: Optional dict from a prior interrupted run. Shape:
            {
                "scenario_order": ["Cat", "Lying", "Stealing"],
                "completed_scenarios": ["Cat"],
                "answers": {...partial dict of 12 keys...},
                "raw_json_by_scenario": {"Cat": "<raw>"},
            }
            If provided, scenarios already in completed_scenarios are skipped.
        on_scenario_done: Optional callback(student_idx, scenario_name, state)
            invoked after each scenario completes. Lets the caller persist
            progress between scenarios so an interruption mid-student doesn't
            lose work.

    Returns:
        (full_answers, state) where state is the full per-student state dict
        suitable for serialization to progress file.
    """
    # Per-student RNG drives scenario order and pair shuffles. This is
    # deterministic given (master_seed, student_idx), so a resumed run will
    # produce identical scenario ordering and pair seeds.
    student_rng = random.Random(f"{master_seed}_{student_idx}")
    scenario_order = ["Cat", "Lying", "Stealing"]
    student_rng.shuffle(scenario_order)

    # Pre-generate all pair_seeds so they're stable even when we skip scenarios
    # on resume. Without this, skipping a completed scenario would advance the
    # RNG differently than on the first run, changing the seeds for later
    # scenarios.
    pair_seeds = [student_rng.randint(0, 10**9) for _ in scenario_order]

    # Initialize state — either fresh or from a partial prior run
    if existing_state is None:
        state = {
            "scenario_order": scenario_order,
            "completed_scenarios": [],
            "answers": {},
            "raw_json_by_scenario": {},
        }
    else:
        state = existing_state
        if state.get("scenario_order") != scenario_order:
            print(f"  WARNING: stored scenario_order does not match regenerated "
                  f"order for student {student_idx}. Discarding partial state "
                  f"and starting this student fresh.")
            state = {
                "scenario_order": scenario_order,
                "completed_scenarios": [],
                "answers": {},
                "raw_json_by_scenario": {},
            }

    for scenario_name, pair_seed in zip(scenario_order, pair_seeds):
        if scenario_name in state["completed_scenarios"]:
            continue  # already done in a prior run

        _, user_payload, expected_keys = build_single_scenario_prompt(
            level, scenario_name, format_type, pair_seed,
            experience_level=experience_level,
        )

        raw = call_api(user_payload)
        parsed = parse_response(raw, expected_keys)
        state["answers"].update(parsed)
        state["raw_json_by_scenario"][scenario_name] = raw
        state["completed_scenarios"].append(scenario_name)

        if on_scenario_done is not None:
            on_scenario_done(student_idx, scenario_name, state)

        time.sleep(0.3)

    full_answers = {k: state["answers"].get(k, None) for k in ALL_KEYS}
    return full_answers, state


# -----------------------------------------------------------------------------
# Pipeline runner
# -----------------------------------------------------------------------------
def _atomic_save_progress(payload, path=PROGRESS_FILE):
    """Write progress JSON atomically (temp file + rename) so an interrupted
    write can't corrupt the file. The os.replace is atomic on POSIX."""
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(payload, f)
    os.replace(tmp_path, path)


def _atomic_save_csv(df, path):
    """Same pattern for the output CSV."""
    tmp_path = path + ".tmp"
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, path)


def run_scenario_at_a_time_pipeline(total_n: int = 96, master_seed: int = 53,
                                    level_proportions=None,
                                    output_csv: str = None,
                                    resume: bool = True):
    """Run the scenario-at-a-time pipeline.

    Default total_n=96 → 3 students per (waypoint x format x experience) cell
    given 4 waypoints x 2 formats x 4 experience levels = 32 cells.
    For uniform balancing, total_n should be a multiple of 32.

    Cell sizes at common N values (with 32 cells):
        N=32  → 1 per cell  (technically uniform but useless for analysis)
        N=64  → 2 per cell  (very thin)
        N=96  → 3 per cell  (recommended minimum)
        N=128 → 4 per cell
        N=160 → 5 per cell  (smallest size where cell-level patterns are readable)

    Resume behavior:
      - If a progress file exists matching all assignment vectors, it's
        loaded and generation picks up where it left off.
      - Resume granularity is per-scenario: an interruption mid-student
        loses at most one in-flight scenario call, not the whole student.
      - If progress params don't match, the file is ignored and a fresh
        run begins. The script does NOT delete the old file in that case.
    """
    if level_proportions is None:
        level_proportions = {"WAYPOINT0": 0.25, "WAYPOINT1": 0.25, "WAYPOINT2": 0.25, "WAYPOINT3": 0.25}
    if output_csv is None:
        output_csv = os.path.join(OUTPUT_DIR, "IIR_TEST_1.csv")

    # Sanity warnings about cell sizes — fire BEFORE any API calls so the
    # user can abort if the design isn't what they wanted.
    n_waypoints = len(level_proportions)
    n_formats = 2
    n_exp = 4
    n_cells = n_waypoints * n_formats * n_exp  # 32 with defaults
    if total_n % n_cells != 0:
        print(f"WARNING: total_n={total_n} is not a multiple of {n_cells} "
              f"({n_waypoints} waypoints x {n_formats} formats x {n_exp} "
              f"experience levels). Some cells will have one more student "
              f"than others (within-cell imbalance of at most 1).")
    cell_size = total_n // n_cells
    if cell_size < 2:
        print(f"WARNING: cell size = {cell_size} student(s). Below 2 per cell "
              f"the design is effectively unbalanced for analysis. "
              f"Recommend total_n >= {n_cells * 2}.")

    # ---- Build deterministic level + format assignments ----
    rng = random.Random(master_seed)

    level_assignments = []
    for lvl, prop in level_proportions.items():
        n = round(total_n * prop)
        level_assignments.extend([lvl] * n)
    while len(level_assignments) < total_n:
        level_assignments.append(list(level_proportions.keys())[-1])
    level_assignments = level_assignments[:total_n]

    format_assignments = [None] * total_n
    for lvl in level_proportions.keys():
        idxs = [i for i, l in enumerate(level_assignments) if l == lvl]
        n_lvl = len(idxs)
        n_comic = n_lvl // 2
        formats = ["comic"] * n_comic + ["text"] * (n_lvl - n_comic)
        rng.shuffle(formats)
        for idx, fmt in zip(idxs, formats):
            format_assignments[idx] = fmt

    # ---- Experience-level assignments ----
    # Balanced uniformly within each (waypoint, format) cell across 4 levels.
    # With cell_size = 3 (n=96), the split is 1/1/1/0 → one level gets dropped
    # in each cell. With cell_size = 4 (n=128), it's a perfect 1/1/1/1.
    # With cell_size = 8, perfect 2/2/2/2. Etc.
    #
    # The assignment is deterministic given master_seed because we use the
    # same `rng` that was already seeded above.
    EXP_LEVELS = ["none", "little", "moderate", "lot"]
    experience_assignments = [None] * total_n
    for lvl in level_proportions.keys():
        for fmt in ("comic", "text"):
            idxs = [
                i for i in range(total_n)
                if level_assignments[i] == lvl and format_assignments[i] == fmt
            ]
            n_cell = len(idxs)
            # Build a balanced pool: as even a split across 4 levels as possible.
            base = n_cell // 4
            remainder = n_cell - 4 * base
            pool = []
            for j, lvl_name in enumerate(EXP_LEVELS):
                # Distribute the remainder across the first `remainder` levels.
                # To avoid systematically favoring certain experience levels
                # when n_cell isn't a multiple of 4, shuffle which levels get
                # the +1 across cells. We do this by picking `remainder` levels
                # at random from EXP_LEVELS for this specific cell.
                pass
            # Pick which levels get the extra student in this cell
            extra_levels = rng.sample(EXP_LEVELS, remainder) if remainder else []
            for lvl_name in EXP_LEVELS:
                count = base + (1 if lvl_name in extra_levels else 0)
                pool.extend([lvl_name] * count)
            rng.shuffle(pool)
            for idx, exp in zip(idxs, pool):
                experience_assignments[idx] = exp

    # ---- Resume logic ----
    # student_states is a list of per-student state dicts (None until a student
    # has at least one scenario completed). This is what gets persisted to
    # the progress file.
    student_states = [None] * total_n
    start_i = 0  # Index of the first student that needs *any* work

    if resume and os.path.exists(PROGRESS_FILE):
        try:
            with open(PROGRESS_FILE, "r") as f:
                prog = json.load(f)
        except (json.JSONDecodeError, OSError) as e:
            print(f"==> Progress file unreadable ({e}). Starting fresh.")
            prog = None

        if prog is not None and (
            prog.get("total_n") == total_n
            and prog.get("master_seed") == master_seed
            and prog.get("level_assignments") == level_assignments
            and prog.get("format_assignments") == format_assignments
            and prog.get("experience_assignments") == experience_assignments
        ):
            student_states = prog["student_states"]
            # Find the first incomplete student. A student is "complete" if
            # all 3 scenarios are in completed_scenarios.
            for idx, state in enumerate(student_states):
                if state is None or len(state.get("completed_scenarios", [])) < 3:
                    start_i = idx
                    break
            else:
                start_i = total_n  # everything done
            n_complete = sum(
                1 for s in student_states
                if s is not None and len(s.get("completed_scenarios", [])) == 3
            )
            print(f"==> Resuming from progress file: "
                  f"{n_complete}/{total_n} students fully complete, "
                  f"resuming at student index {start_i}.")
        else:
            print("==> Progress file found but parameters differ from current "
                  "run. Starting fresh. (Old file not deleted — remove it "
                  f"manually at {PROGRESS_FILE} if you want to discard it.)")

    # ---- Sanity report ----
    print(f"Level distribution: "
          f"{pd.Series(level_assignments).value_counts().to_dict()}")
    df_verify = pd.DataFrame({"Level": level_assignments,
                              "Format": format_assignments,
                              "Experience": experience_assignments})
    print("Format breakdown per level:")
    print(df_verify.groupby(['Level', 'Format']).size().unstack(fill_value=0).to_dict('index'))
    print("Experience breakdown per (level, format):")
    print(df_verify.groupby(['Level', 'Format', 'Experience']).size().unstack(fill_value=0))

    # ---- Helper: persist progress after each scenario ----
    def save_progress():
        _atomic_save_progress({
            "total_n": total_n,
            "master_seed": master_seed,
            "level_assignments": level_assignments,
            "format_assignments": format_assignments,
            "experience_assignments": experience_assignments,
            "student_states": student_states,
        })

    def on_scenario_done(student_idx, scenario_name, state):
        student_states[student_idx] = state
        save_progress()

    # ---- Main generation loop ----
    if start_i >= total_n:
        print("All responses already generated. Skipping API calls.")
    else:
        pipeline_start = time.time()
        student_times = []

        for i in range(start_i, total_n):
            level = level_assignments[i]
            fmt = format_assignments[i]
            exp = experience_assignments[i]
            n_done = (
                len(student_states[i]["completed_scenarios"])
                if student_states[i] is not None else 0
            )
            elapsed_total = time.time() - pipeline_start
            elapsed_str = time.strftime("%H:%M:%S", time.gmtime(elapsed_total))
            base_msg = (f"[{i+1}/{total_n}] R{i+1:04d} level={level} "
                        f"format={fmt} experience={exp} | elapsed={elapsed_str}")
            print(base_msg + (f" (resuming with {n_done}/3 scenarios done)"
                              if n_done > 0 else ""))

            student_start = time.time()
            try:
                answers, state = generate_one_student(
                    i, level, fmt, master_seed,
                    experience_level=exp,
                    existing_state=student_states[i],
                    on_scenario_done=on_scenario_done,
                )
                student_states[i] = state
            except Exception as e:
                print(f"\n  FAILED on student {i+1}: {e}")
                save_progress()
                raise

            student_elapsed = time.time() - student_start
            student_times.append(student_elapsed)
            students_left = total_n - i - 1
            avg_time = sum(student_times) / len(student_times)
            eta_seconds = avg_time * students_left
            eta_str = time.strftime("%H:%M:%S", time.gmtime(eta_seconds))
            total_elapsed_str = time.strftime("%H:%M:%S", time.gmtime(time.time() - pipeline_start))
            print(f"  -> done in {student_elapsed:.1f}s | total elapsed={total_elapsed_str} | ETA={eta_str} ({students_left} students left)")

    # ---- Build final CSV from student_states ----
    rows = []
    for i in range(total_n):
        state = student_states[i]
        if state is None:
            answers = {k: None for k in ALL_KEYS}
            scenario_order = []
        else:
            answers = {k: state["answers"].get(k, None) for k in ALL_KEYS}
            scenario_order = state.get("scenario_order", [])

        rows.append({
            "respondent_id": f"R{i+1:04d}",
            "profile_level": level_assignments[i],
            "format": format_assignments[i],
            "experience": experience_assignments[i],
            "scenario_order": "|".join(scenario_order),
            **answers,
        })

    df = pd.DataFrame(rows)
    _atomic_save_csv(df, output_csv)

    # Clean up progress file only on a fully successful run
    n_complete = sum(
        1 for s in student_states
        if s is not None and len(s.get("completed_scenarios", [])) == 3
    )
    if n_complete == total_n and os.path.exists(PROGRESS_FILE):
        os.remove(PROGRESS_FILE)
        print(f"All {total_n} students complete. Progress file removed.")

    print(f"\nSaved {len(rows)} respondents to {output_csv}")
    return df


if __name__ == "__main__":
    # Default: 96 students = 3 per (waypoint x format x experience) cell
    #        = ~288 API calls total (96 students x 3 scenarios each).
    # Multiples of 32 give uniform balancing across all design cells.
    # Adjust total_n if you want a larger or smaller run.
    run_scenario_at_a_time_pipeline(total_n=32, master_seed=53)

==> Resuming from progress file: 5/32 students fully complete, resuming at student index 5.
Level distribution: {'IIR0': 8, 'IIR1': 8, 'IIR2': 8, 'IIR3': 8}
Format breakdown per level:
{'IIR0': {'comic': 4, 'text': 4}, 'IIR1': {'comic': 4, 'text': 4}, 'IIR2': {'comic': 4, 'text': 4}, 'IIR3': {'comic': 4, 'text': 4}}
Experience breakdown per (level, format):
Experience    little  lot  moderate  none
Level Format                             
IIR0  comic        1    1         1     1
      text         1    1         1     1
IIR1  comic        1    1         1     1
      text         1    1         1     1
IIR2  comic        1    1         1     1
      text         1    1         1     1
IIR3  comic        1    1         1     1
      text         1    1         1     1
[6/32] R0006 level=IIR0 format=text experience=none | elapsed=00:00:00


c:\Users\david\Documents\IIR\.venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


  -> done in 54.5s | total elapsed=00:00:54 | ETA=00:23:37 (26 students left)
[7/32] R0007 level=IIR0 format=text experience=moderate | elapsed=00:00:54
  -> done in 65.5s | total elapsed=00:02:00 | ETA=00:25:00 (25 students left)
[8/32] R0008 level=IIR0 format=comic experience=lot | elapsed=00:02:00
  -> done in 69.7s | total elapsed=00:03:09 | ETA=00:25:18 (24 students left)
[9/32] R0009 level=IIR1 format=text experience=lot | elapsed=00:03:09
  -> done in 41.3s | total elapsed=00:03:51 | ETA=00:22:08 (23 students left)
[10/32] R0010 level=IIR1 format=text experience=none | elapsed=00:03:51
  -> done in 44.5s | total elapsed=00:04:35 | ETA=00:20:12 (22 students left)
[11/32] R0011 level=IIR1 format=comic experience=none | elapsed=00:04:35
  -> done in 44.1s | total elapsed=00:05:19 | ETA=00:18:38 (21 students left)
[12/32] R0012 level=IIR1 format=comic experience=moderate | elapsed=00:05:19
  -> done in 59.7s | total elapsed=00:06:19 | ETA=00:18:04 (20 students left)
[13/32] R0013 le